In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score


In [36]:
df = pd.read_csv("../data/processed/high_value_churn_data.csv")

In [37]:
X = df.drop("churn", axis=1)
y = df["churn"]

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [40]:
y_train.value_counts(normalize=True)

churn
0    0.913612
1    0.086388
Name: proportion, dtype: float64

In [41]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
###Apply PCA

In [43]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original features:", X_train.shape[1])
print("Reduced features:", X_train_pca.shape[1])

Original features: 167
Reduced features: 73


In [44]:
### Handle Balance

In [45]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_pca, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [46]:
### Evaluate Model

In [47]:
y_pred = model.predict(X_test_pca)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test_pca)[:,1]))

              precision    recall  f1-score   support

           0       0.98      0.81      0.88      5484
           1       0.29      0.83      0.43       519

    accuracy                           0.81      6003
   macro avg       0.63      0.82      0.66      6003
weighted avg       0.92      0.81      0.84      6003

ROC-AUC: 0.889516393108556


In [38]:
[col for col in df.columns if '_9' in col]

[]

In [48]:
### threshold tuning

In [49]:
y_prob = model.predict_proba(X_test_pca)[:,1]

# try threshold = 0.7
y_pred_07 = (y_prob >= 0.7).astype(int)

print(classification_report(y_test, y_pred_07))

              precision    recall  f1-score   support

           0       0.97      0.92      0.94      5484
           1       0.44      0.68      0.53       519

    accuracy                           0.90      6003
   macro avg       0.70      0.80      0.74      6003
weighted avg       0.92      0.90      0.91      6003



In [50]:
y_prob = model.predict_proba(X_test_pca)[:,1]

# try threshold = 0.6
y_pred_06 = (y_prob >= 0.6).astype(int)

print(classification_report(y_test, y_pred_06))

              precision    recall  f1-score   support

           0       0.98      0.87      0.92      5484
           1       0.37      0.78      0.50       519

    accuracy                           0.87      6003
   macro avg       0.67      0.83      0.71      6003
weighted avg       0.92      0.87      0.89      6003



In [51]:
y_prob = model.predict_proba(X_test_pca)[:,1]

# try threshold = 0.8
y_pred_08 = (y_prob >= 0.8).astype(int)

print(classification_report(y_test, y_pred_08))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5484
           1       0.50      0.50      0.50       519

    accuracy                           0.91      6003
   macro avg       0.73      0.73      0.73      6003
weighted avg       0.91      0.91      0.91      6003

